# Preprocesamiento del dataset

Este notebook prepara el dataset de abandono de clientes para el
entrenamiento y evaluación de modelos de aprendizaje automático.

El proceso incluye la separación de variables predictoras y objetivo,
la división estratificada de los datos y la transformación de variables
numéricas y categóricas.

Las transformaciones se ajustarán exclusivamente con los datos de
entrenamiento para evitar filtración de información.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 1. Carga del dataset procesado

Se carga el dataset generado durante la etapa de evaluación de calidad y
análisis exploratorio.

In [2]:
ruta = "../data/processed/customer_churn_clean.csv"

df = pd.read_csv(
    ruta,
    keep_default_na=False,
    na_values=[""]
)

print("Dimensiones:", df.shape)
df.head()

Dimensiones: (10000, 32)


,customer_id,gender,age,country,city,customer_segment,tenure_months,signup_channel,contract_type,monthly_logins,...,avg_resolution_time,complaint_type,csat_score,escalations,email_open_rate,marketing_click_rate,nps_score,survey_response,referral_count,churn
0,CUST_00001,Male,68,Bangladesh,London,SME,22,Web,Monthly,26,...,13.354360,Service,4.0,0,0.71,0.40,27,Satisfied,1,0
1,CUST_00002,Female,57,Canada,Sydney,Individual,9,Mobile,Monthly,7,...,25.140088,Billing,2.0,0,0.78,0.33,-19,Neutral,2,1
2,CUST_00003,Male,24,Germany,New York,SME,58,Web,Yearly,19,...,27.572928,Service,3.0,0,0.35,0.49,80,Neutral,1,0
3,CUST_00004,Male,49,Australia,Dhaka,Individual,19,Mobile,Yearly,34,...,26.420822,Technical,5.0,1,0.83,0.15,100,Neutral,0,0
4,CUST_00005,Male,65,Bangladesh,Delhi,Individual,52,Web,Monthly,20,...,26.674579,Technical,4.0,0,0.65,0.44,21,Unsatisfied,1,0


In [3]:
print("Valores nulos:", df.isnull().sum().sum())
print("Duplicados:", df.duplicated().sum())
print("Distribución de churn:")

display(
    df["churn"]
    .value_counts()
    .sort_index()
    .to_frame("cantidad")
)


Valores nulos: 0
Duplicados: 0
Distribución de churn:


,cantidad
churn,
0,8979
1,1021


## 2. Separación de variables predictoras y variable objetivo

La variable `churn` corresponde al objetivo que se desea predecir.

Las demás columnas serán utilizadas como variables predictoras, con la
excepción de `customer_id`, que se elimina por ser únicamente un
identificador y no representar una característica real del cliente.

In [4]:
X = df.drop(
    columns=[
        "churn",
        "customer_id"
    ]
)

y = df["churn"].astype(int)

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

print(
    "\nDistribución porcentual de churn:"
)

display(
    y.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
    .to_frame("porcentaje")
)

Dimensiones de X: (10000, 30)
Dimensiones de y: (10000,)

Distribución porcentual de churn:


,porcentaje
churn,
0,89.79
1,10.21


## 3. División de los datos

Los datos se dividen en tres conjuntos:

- 70 % para entrenamiento.
- 15 % para validación.
- 15 % para prueba.

El conjunto de entrenamiento será utilizado para que los modelos aprendan.
El conjunto de validación permitirá comparar y ajustar los modelos, mientras
que el conjunto de prueba se reservará para la evaluación final.

La división se realiza de forma estratificada para conservar aproximadamente
la misma proporción de clientes que permanecen y abandonan en los tres
conjuntos.

In [5]:
X_train, X_temporal, y_train, y_temporal = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temporal,
    y_temporal,
    test_size=0.50,
    random_state=42,
    stratify=y_temporal
)

In [6]:
print("Conjunto de entrenamiento:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nConjunto de validación:")
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nConjunto de prueba:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

Conjunto de entrenamiento:
X_train: (7000, 30)
y_train: (7000,)

Conjunto de validación:
X_val: (1500, 30)
y_val: (1500,)

Conjunto de prueba:
X_test: (1500, 30)
y_test: (1500,)


In [7]:
def mostrar_distribucion(nombre, datos):
    distribucion = (
        datos.value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )

    print(f"\n{nombre}:")
    print(distribucion)


mostrar_distribucion(
    "Entrenamiento",
    y_train
)

mostrar_distribucion(
    "Validación",
    y_val
)

mostrar_distribucion(
    "Prueba",
    y_test
)


Entrenamiento:
churn
0    89.79
1    10.21
Name: proportion, dtype: float64

Validación:
churn
0    89.8
1    10.2
Name: proportion, dtype: float64

Prueba:
churn
0    89.8
1    10.2
Name: proportion, dtype: float64


### Interpretación de la división

La división generó 7.000 observaciones para entrenamiento, 1.500 para
validación y 1.500 para prueba.

Gracias a la estratificación, los tres conjuntos conservaron una proporción
similar de clientes que permanecen y abandonan. Esto permite que el
entrenamiento y la evaluación se realicen sobre distribuciones comparables.

El conjunto de prueba permanecerá separado y no será utilizado para ajustar
las transformaciones ni los modelos.

### Interpretación de la división

La división generó 7.000 observaciones para entrenamiento, 1.500 para
validación y 1.500 para prueba.

Gracias a la estratificación, los tres conjuntos conservaron una proporción
similar de clientes que permanecen y abandonan. Esto permite que el
entrenamiento y la evaluación se realicen sobre distribuciones comparables.

El conjunto de prueba permanecerá separado y no será utilizado para ajustar
las transformaciones ni los modelos.

In [8]:
variables_numericas = (
    X_train
    .select_dtypes(include=["int64", "float64"])
    .columns
    .tolist()
)

variables_categoricas = (
    X_train
    .select_dtypes(include=["object", "category", "bool"])
    .columns
    .tolist()
)

print("Cantidad de variables numéricas:", len(variables_numericas))
print("Cantidad de variables categóricas:", len(variables_categoricas))
print("Total de variables:", len(variables_numericas) + len(variables_categoricas))

print("\nVariables numéricas:")
print(variables_numericas)

print("\nVariables categóricas:")
print(variables_categoricas)

Cantidad de variables numéricas: 19
Cantidad de variables categóricas: 11
Total de variables: 30

Variables numéricas:
['age', 'tenure_months', 'monthly_logins', 'weekly_active_days', 'avg_session_time', 'features_used', 'usage_growth_rate', 'last_login_days_ago', 'monthly_fee', 'total_revenue', 'payment_failures', 'support_tickets', 'avg_resolution_time', 'csat_score', 'escalations', 'email_open_rate', 'marketing_click_rate', 'nps_score', 'referral_count']

Variables categóricas:
['gender', 'country', 'city', 'customer_segment', 'signup_channel', 'contract_type', 'payment_method', 'discount_applied', 'price_increase_last_3m', 'complaint_type', 'survey_response']


C:\Users\gvera\AppData\Local\Temp\ipykernel_86708\1722193668.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  .select_dtypes(include=["object", "category", "bool"])


In [9]:
columnas_detectadas = (
    variables_numericas +
    variables_categoricas
)

columnas_no_detectadas = [
    columna
    for columna in X_train.columns
    if columna not in columnas_detectadas
]

print(
    "Columnas no detectadas:",
    columnas_no_detectadas
)

Columnas no detectadas: []


### Interpretación de los tipos de variables

Las 30 variables predictoras fueron clasificadas como numéricas o
categóricas. No quedaron columnas sin identificar, por lo que todas podrán
ser incorporadas al proceso de transformación.

### 4.3 Construcción del preprocesador

Se utiliza `ColumnTransformer` para aplicar transformaciones diferentes
según el tipo de variable.

Las variables numéricas serán estandarizadas para que presenten escalas
comparables. Las variables categóricas serán codificadas mediante
`OneHotEncoder`, creando columnas binarias para cada categoría.

El parámetro `handle_unknown="ignore"` permitirá procesar categorías que
puedan aparecer en validación o prueba, aunque no hayan aparecido durante
el entrenamiento.

In [10]:
preprocesador = ColumnTransformer(
    transformers=[
        (
            "numericas",
            StandardScaler(),
            variables_numericas
        ),
        (
            "categoricas",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            variables_categoricas
        )
    ],
    remainder="drop"
)

preprocesador

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numericas', ...), ('categoricas', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``

In [11]:
X_train_preprocesado = preprocesador.fit_transform(
    X_train
)

X_val_preprocesado = preprocesador.transform(
    X_val
)

X_test_preprocesado = preprocesador.transform(
    X_test
)

print(
    "Entrenamiento preprocesado:",
    X_train_preprocesado.shape
)

print(
    "Validación preprocesada:",
    X_val_preprocesado.shape
)

print(
    "Prueba preprocesada:",
    X_test_preprocesado.shape
)

Entrenamiento preprocesado: (7000, 58)
Validación preprocesada: (1500, 58)
Prueba preprocesada: (1500, 58)


### 4.5 Revisión de las variables transformadas

Se obtienen los nombres de las variables generadas por el preprocesador para
verificar la transformación aplicada a las columnas numéricas y categóricas.

Las variables numéricas mantienen una columna por característica, mientras
que las categóricas generan columnas binarias para representar cada una de
sus categorías.

In [12]:
nombres_variables_preprocesadas = (
    preprocesador.get_feature_names_out()
)

print(
    "Cantidad final de variables:",
    len(nombres_variables_preprocesadas)
)

print("\nNombres de las variables transformadas:")

for nombre in nombres_variables_preprocesadas:
    print(nombre)

Cantidad final de variables: 58

Nombres de las variables transformadas:
numericas__age
numericas__tenure_months
numericas__monthly_logins
numericas__weekly_active_days
numericas__avg_session_time
numericas__features_used
numericas__usage_growth_rate
numericas__last_login_days_ago
numericas__monthly_fee
numericas__total_revenue
numericas__payment_failures
numericas__support_tickets
numericas__avg_resolution_time
numericas__csat_score
numericas__escalations
numericas__email_open_rate
numericas__marketing_click_rate
numericas__nps_score
numericas__referral_count
categoricas__gender_Female
categoricas__gender_Male
categoricas__country_Australia
categoricas__country_Bangladesh
categoricas__country_Canada
categoricas__country_Germany
categoricas__country_India
categoricas__country_UK
categoricas__country_USA
categoricas__city_Berlin
categoricas__city_Delhi
categoricas__city_Dhaka
categoricas__city_London
categoricas__city_New York
categoricas__city_Sydney
categoricas__city_Toronto
categoric

### Interpretación del preprocesamiento

El conjunto de entrenamiento quedó compuesto por 7.000 observaciones y
58 variables transformadas. Los conjuntos de validación y prueba contienen
1.500 observaciones cada uno y conservan las mismas 58 variables.

El aumento desde 30 hasta 58 variables se debe a la codificación de las
variables categóricas mediante `OneHotEncoder`, donde cada categoría fue
representada mediante una columna binaria.

Las variables numéricas fueron estandarizadas utilizando exclusivamente los
parámetros aprendidos desde el conjunto de entrenamiento. Posteriormente,
las mismas reglas fueron aplicadas a validación y prueba.

Esto garantiza que los tres conjuntos presenten una estructura compatible y
evita que información de validación o prueba influya en el preprocesamiento.

## 5. Revisión del desbalance en el conjunto de entrenamiento

La variable objetivo presenta un desbalance importante entre clientes que
permanecen y clientes que abandonan.

Se revisa la distribución dentro del conjunto de entrenamiento para definir
posteriormente estrategias que permitan reducir el efecto de este
desbalance durante el modelamiento.

In [13]:
distribucion_entrenamiento = (
    y_train
    .value_counts()
    .sort_index()
)

porcentaje_entrenamiento = (
    y_train
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

resumen_entrenamiento = pd.DataFrame({
    "cantidad": distribucion_entrenamiento,
    "porcentaje": porcentaje_entrenamiento
})

resumen_entrenamiento = resumen_entrenamiento.rename(
    index={
        0: "Permanece (0)",
        1: "Abandona (1)"
    }
)

resumen_entrenamiento

,cantidad,porcentaje
churn,,
Permanece (0),6285,89.79
Abandona (1),715,10.21


In [14]:
cantidad_permanece = (y_train == 0).sum()
cantidad_abandona = (y_train == 1).sum()

relacion_desbalance = (
    cantidad_permanece /
    cantidad_abandona
)

print("Clientes que permanecen:", cantidad_permanece)
print("Clientes que abandonan:", cantidad_abandona)

print(
    f"Relación de desbalance: "
    f"{relacion_desbalance:.2f} a 1"
)

Clientes que permanecen: 6285
Clientes que abandonan: 715
Relación de desbalance: 8.79 a 1


### 5.1 Estrategia inicial frente al desbalance

El conjunto de entrenamiento contiene muchos más clientes que permanecen
que clientes que abandonan.

Como estrategia inicial se utilizarán pesos de clase. Esto permite asignar
una mayor importancia a los errores cometidos sobre la clase minoritaria,
sin eliminar registros ni crear observaciones artificiales.

Los pesos se calcularán exclusivamente a partir del conjunto de
entrenamiento. Los conjuntos de validación y prueba conservarán su
distribución original.

In [15]:
from sklearn.utils.class_weight import compute_class_weight

In [16]:
clases = np.array([0, 1])

pesos_calculados = compute_class_weight(
    class_weight="balanced",
    classes=clases,
    y=y_train
)

pesos_clase = {
    int(clase): float(peso)
    for clase, peso in zip(
        clases,
        pesos_calculados
    )
}

print("Pesos calculados:")
print(pesos_clase)

Pesos calculados:
{0: 0.5568814638027049, 1: 4.895104895104895}


### 5.2 Prevención de filtración durante el balanceo

Las estrategias de balanceo se aplicarán exclusivamente durante el
entrenamiento de los modelos.

Los conjuntos de validación y prueba mantendrán la distribución original
del churn, ya que deben representar de forma realista el comportamiento
del dataset.

Aplicar balanceo antes de dividir los datos o modificar los conjuntos de
validación y prueba podría producir filtración de información y una
evaluación poco realista.

Por lo tanto:

- Entrenamiento: se podrán aplicar pesos de clase u otras estrategias.
- Validación: no se balanceará.
- Prueba: no se balanceará.

Más adelante se compararán modelos sin ajuste de clases, utilizando
`class_weight=None`, con modelos compensados mediante
`class_weight="balanced"`.

## 6. Verificación final

Se comprueba que los conjuntos preprocesados mantengan la cantidad correcta
de observaciones, posean el mismo número de variables y no contengan valores
nulos o infinitos.

Esta validación permite confirmar que los datos están correctamente
preparados antes del entrenamiento de los modelos.

In [17]:
def revisar_matriz(nombre, matriz):
    matriz_revision = (
        matriz.toarray()
        if hasattr(matriz, "toarray")
        else matriz
    )

    print(f"\n{nombre}")
    print("Dimensiones:", matriz.shape)
    print(
        "Valores nulos:",
        np.isnan(matriz_revision).sum()
    )
    print(
        "Valores infinitos:",
        np.isinf(matriz_revision).sum()
    )


revisar_matriz(
    "Entrenamiento",
    X_train_preprocesado
)

revisar_matriz(
    "Validación",
    X_val_preprocesado
)

revisar_matriz(
    "Prueba",
    X_test_preprocesado
)


Entrenamiento
Dimensiones: (7000, 58)
Valores nulos: 0
Valores infinitos: 0

Validación
Dimensiones: (1500, 58)
Valores nulos: 0
Valores infinitos: 0

Prueba
Dimensiones: (1500, 58)
Valores nulos: 0
Valores infinitos: 0


## 7. Conclusiones del preprocesamiento

El dataset fue separado entre variables predictoras (`X`) y variable
objetivo (`y`). Las columnas `customer_id` y `churn` fueron excluidas de
las variables predictoras, debido a que corresponden respectivamente a un
identificador y al resultado que se desea predecir.

Los datos fueron divididos de manera estratificada en 70 % para
entrenamiento, 15 % para validación y 15 % para prueba. Esto permitió
conservar una proporción similar de clientes que permanecen y abandonan en
los tres conjuntos.

Las variables numéricas fueron estandarizadas mediante `StandardScaler`,
mientras que las variables categóricas fueron transformadas mediante
`OneHotEncoder`.

El preprocesador fue ajustado exclusivamente con el conjunto de
entrenamiento y posteriormente aplicado a validación y prueba, evitando
filtración de información.

Después de las transformaciones, las 30 variables predictoras originales
generaron 58 variables. El conjunto de entrenamiento quedó compuesto por
7.000 observaciones, mientras que validación y prueba quedaron conformados
por 1.500 observaciones cada uno.

Los tres conjuntos conservaron la misma estructura y no presentaron valores
nulos ni infinitos.

Debido al desbalance de la variable objetivo, se calcularon pesos de clase
que podrán utilizarse durante el entrenamiento de los modelos. Las
estrategias de balanceo se aplicarán únicamente al conjunto de entrenamiento,
mientras que validación y prueba mantendrán su distribución original.

Con estas transformaciones, los datos quedaron preparados para el
entrenamiento y evaluación de modelos de clasificación.